# Tutorial: retrieve a Gaia DR3 astrometric catalog

This notebook queries the Gaia DR3 archive and converts the result into a `warpfield.AstrometricCatalog`.

We retrieve sources within one degree of $(\mathrm{ra}, \mathrm{dec}) = (269.267^\circ, -18.985^\circ)$ and require a parallax signal-to-noise ratio larger than 10. The resulting catalog contains ICRS positions, proper motions, parallaxes, their uncertainties, G-band photometry, and reference epochs.

## 1. Define the search region

`query_gaia()` accepts a scalar `SkyCoord` as the search center. The radius may be an Astropy angle or a number interpreted in degrees.

In [1]:
from astropy.coordinates import SkyCoord
import astropy.units as u

from warpfield import AstrometricCatalog
from warpfield.query import query_gaia

In [2]:
center = SkyCoord(ra=269.267 * u.deg, dec=-18.985 * u.deg, frame='icrs')
radius = 1.0 * u.deg

## 2. Query Gaia DR3

`query_gaia()` sends an asynchronous ADQL query through `astroquery`. By default it uses `gaiadr3.gaia_source`, applies `parallax_over_error > 10`, rejects rows with missing proper motion, parallax, or G-band photometry, and returns all matching rows. The G-band magnitude uncertainty is calculated from `phot_g_mean_flux_over_error`.

The query may take about a minute for this field. During development, pass a positive `row_limit` such as `row_limit=100` to request a smaller sample.

In [4]:
catalog = query_gaia(
  center,
  radius,
  snr_limit=30.0,
  row_limit=-1,
)

INFO: Query finished. [astroquery.utils.tap.core]


The result is an immutable `AstrometricCatalog`, not a raw Gaia table or `SkyCoord`. Its arrays retain Astropy units, and Gaia's `pmra` column is exposed as `pm_ra_cosdec` to make the coordinate convention explicit.

In [5]:
print(type(catalog))
print(f'{len(catalog):,} sources')
assert isinstance(catalog, AstrometricCatalog)

<class 'warpfield.catalog.astrometric.AstrometricCatalog'>
4,478 sources


## 3. Inspect and select catalog entries

The main attributes are `ra`, `dec`, `pm_ra_cosdec`, `pm_dec`, `parallax`, `magnitude`, and `epoch`. Their optional uncertainty attributes use the `_error` suffix. Indexing preserves the catalog dimension, so `catalog[0]` is a one-source `AstrometricCatalog`. Slices and iteration follow the same convention.

In [6]:
first_source = catalog[0]

print('RA:', first_source.ra)
print('Dec:', first_source.dec)
print('Proper motion in RA:', first_source.pm_ra_cosdec)
print('Proper motion in Dec:', first_source.pm_dec)
print('Parallax:', first_source.parallax)
print('Magnitude:', first_source.magnitude)
print('Magnitude error:', first_source.magnitude_error)
print('RA error:', first_source.ra_error)
print('Dec error:', first_source.dec_error)
print('Proper-motion error in RA:', first_source.pm_ra_cosdec_error)
print('Proper-motion error in Dec:', first_source.pm_dec_error)
print('Parallax error:', first_source.parallax_error)
print('Reference epoch:', first_source.epoch)

RA: [269d44m53.65603652s]
Dec: [-19d49m29.73147431s]
Proper motion in RA: [-6.41567453] mas / yr
Proper motion in Dec: [-5.18055464] mas / yr
Parallax: [1.35439672] mas
Reference epoch: [2016.]


`catalog.skycoord` constructs the corresponding ICRS `SkyCoord`, including distance, proper motion, and observation epoch. This representation is useful when applying Astropy coordinate transformations.

In [7]:
catalog.skycoord[:3]

<SkyCoord (ICRS): (ra, dec, distance) in (deg, deg, pc)
    [(269.74823779, -19.82492541, 738.33610461),
     (269.76693285, -19.81864425, 552.88252058),
     (269.72521833, -19.76488103, 365.1188266 )]
 (pm_ra_cosdec, pm_dec) in mas / yr
    [( -6.41567453,  -5.18055464), (  4.13454382,  -8.57718228),
     (-14.13944986, -41.89080425)]>

## 4. Convert to a QTable

`to_qtable()` exports the available astrometric, uncertainty, and photometric attributes to a unit-aware `QTable`. Optional attributes are omitted when they are `None`. The generated `source_id` is a zero-origin row identifier used by warpfield; it is not the original Gaia DR3 `source_id`.

In [8]:
table = catalog.to_qtable()
table[:5]

source_id,ra,dec,pm_ra_cosdec,pm_dec,parallax,epoch
,deg,deg,mas / yr,mas / yr,mas,
int64,float64,float64,float64,float64,float64,Time
0,269.7482377879231,-19.824925409530582,-6.4156745315716295,-5.180554640433498,1.3543967222515656,2016.0
1,269.76693284761717,-19.818644254970298,4.134543822713832,-8.57718228379656,1.8087025051099956,2016.0
2,269.7252183318344,-19.764881026579314,-14.139449863615598,-41.89080424707823,2.7388343934781947,2016.0
3,269.75660041103276,-19.731278978848742,-4.271982014581065,-4.27828145640077,1.2658627053735652,2016.0
4,269.82180094801686,-19.656104866180637,0.5500894928456234,0.29661192945040943,0.6707421343259654,2016.0


## 5. Save and restore the catalog

Astropy's ECSV format preserves the QTable units and the `Time` mixin column needed for a lossless round trip. The saved table can therefore be passed directly to `AstrometricCatalog.from_qtable()` after reading.

In [9]:
table.write('gaia_dr3.ecsv', overwrite=True)

In [10]:
from astropy.table import QTable

restored = AstrometricCatalog.from_qtable(QTable.read('gaia_dr3.ecsv'))
print(f'Restored {len(restored):,} sources')

Restored 4,478 sources


The saved catalog is now ready for later propagation to an observing epoch and observer frame. That step converts the reference-epoch `AstrometricCatalog` into a `SourceCatalog` of apparent positions.